# Kaggle Submission — DistilBERT Model
**F20AA CW2 — Google Maps Review Challenge**

Generates Kaggle-ready CSV using the fine-tuned DistilBERT model saved in `distilbert_saved/`.

## 1. Imports

In [1]:
import os
import re
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('Imports done.')

Using device: cpu
Imports done.


## 2. Load Kaggle Test Data

In [2]:
print('Loading Kaggle test data...')
test = pd.read_csv('data/kaggle_test.csv')
test.columns = [c.strip().lower() for c in test.columns]
print(f'Test shape: {test.shape}')
test.head()

Loading Kaggle test data...
Test shape: (89100, 2)


,id,text
0,1,Quite easy to rent a car bur it is not easy to...
1,2,Nice voleyball court close to restaurants and ...
2,3,"Very nice built homes, the future locations ar..."
3,4,This dental clinic appears to be friendly with...
4,5,We came in to discuss tattoos. Only person th...


## 3. Preprocess & Tokenise

Minimal cleaning (HTML + URL removal only) — same as in `DistelBert_Model.ipynb`.

In [3]:
BERT_MAX_LEN = 128

# Use the Downloads copy — the distilbert_saved in the project folder is a truncated copy
MODEL_PATH = os.path.expanduser('~/Downloads/distilbert_saved')

def bert_preprocess(text):
    text = str(text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    return text.strip()

test_texts = test['text'].apply(bert_preprocess).tolist()
print(f'Test samples: {len(test_texts):,}')
print(f'Model path:   {MODEL_PATH}')

Test samples: 89,100
Model path:   /Users/muhammadsaad/Downloads/distilbert_saved


In [4]:
print('Loading tokenizer from saved model...')
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_PATH)

class TestDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors='pt'
        )

    def __len__(self):
        return self.encodings['input_ids'].shape[0]

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

print('Tokenizing test set...')
test_dataset = TestDataset(test_texts, tokenizer, BERT_MAX_LEN)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)
print(f'Test batches: {len(test_loader)}')

Loading tokenizer from saved model...
Tokenizing test set...
Test batches: 1393


## 4. Load Saved DistilBERT Model

In [5]:
print(f'Loading DistilBERT model from {MODEL_PATH}...')
bert_model = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH)
bert_model = bert_model.to(device)
bert_model.eval()
print('Model loaded.')

Loading DistilBERT model from /Users/muhammadsaad/Downloads/distilbert_saved...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded.


## 5. Generate Predictions

In [6]:
print('Generating predictions...')
all_preds = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs        = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        preds          = outputs.logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())

predictions = np.array(all_preds) + 1  # convert 0-4 → 1-5

print(f'Predictions shape: {predictions.shape}')
print('\nPrediction distribution:')
print(pd.Series(predictions).value_counts().sort_index())

Generating predictions...
Predictions shape: (89100,)

Prediction distribution:
1    18536
2     7855
3    14966
4    37956
5     9787
Name: count, dtype: int64


## 6. Save Submission CSV

In [7]:
id_col = 'id' if 'id' in test.columns else test.columns[0]

submission = pd.DataFrame({
    'Id':     test[id_col].values,
    'Rating': predictions
})

assert submission.shape[1] == 2
assert list(submission.columns) == ['Id', 'Rating']
assert submission['Rating'].between(1, 5).all()
assert submission['Id'].is_unique
assert not submission.isnull().any().any()

print(f'All checks passed!')
print(f'Rows: {len(submission):,}')
submission.head(10)

All checks passed!
Rows: 89,100


,Id,Rating
0,1,4
1,2,4
2,3,4
3,4,2
4,5,1
5,6,4
6,7,4
7,8,5
8,9,1
9,10,4


In [8]:
os.makedirs('data/kaggle_submission', exist_ok=True)
output_path = 'data/kaggle_submission/submission_DistilBERT.csv'
submission.to_csv(output_path, index=False)
print(f'Saved to: {output_path}')

verify = pd.read_csv(output_path)
print(f'Verified rows: {len(verify):,}')
print(verify.head())

Saved to: data/kaggle_submission/submission_DistilBERT.csv
Verified rows: 89,100
   Id  Rating
0   1       4
1   2       4
2   3       4
3   4       2
4   5       1
